# APID 0 exploration

APID 0 is at the bottom of the list: 27 packets in the 13 dumps. APID 0 is not special in CCSDS (APID 2047 is the
idle one), but small APIDs are worth a look because they are often a service channel (time, events, test data).
Questions:

1. How many of the 27 packets are real packets and how many are artifacts of the frame / packet reassembly?
2. What is in the real ones?
3. Anything LYRA, or the position of the Sun?

In [1]:
from collections import Counter, defaultdict
from contextlib import suppress
from dataclasses import dataclass
from datetime import UTC, datetime, timedelta
from pathlib import Path

import numpy as np

from cadu import iter_frames, iter_packets, packet_apid, packet_sequence_count

In [2]:
CADU_DIR = Path("cadus")
APID = 0

SEQUENCE_MODULO = 1 << 14
VALID_VIRTUAL_CHANNELS = (0, 1, 2, 3, 7)  # decided in section 2
TEXT_HEADER = bytes.fromhex("100889")  # secondary header of the APID 18 text packets, see APID_18.ipynb
GPS_EPOCH = datetime(1980, 1, 6, tzinfo=UTC)
SECONDS_PER_WEEK = 604800
MAX_TICK_SPREAD = 10_000_000  # packets reassembled across lost frames have garbage where the time should be

In [3]:
def crc16(data: bytes) -> int:
    """CRC-16/CCITT-FALSE."""
    crc = 0xFFFF
    for byte in data:
        crc ^= byte << 8
        for _ in range(8):
            crc = ((crc << 1) ^ 0x1021) & 0xFFFF if crc & 0x8000 else (crc << 1) & 0xFFFF
    return crc

## 1. Load

One pass over the dumps: the frame headers (to see which virtual channels are real), the APIDs of all packets, the
APID 0 packets, and the GPS time and tick of the APID 18 `F4` records (to date the APID 0 packets).

In [4]:
frame_headers = Counter()  # (vcid, first two bytes of the frame without the virtual channel id)
apids = Counter()
packets: list[tuple[str, int, bytes]] = []  # (file name, vcid, packet)
text_ticks: dict[str, list[int]] = defaultdict(list)  # APID 18 packets: tick counters
gps_minus_tick = []  # week * seconds per week + second of the week - tick, from the F4 records

for path in sorted(CADU_DIR.glob("*.cadu")):
    for frame in iter_frames(path):
        frame_headers[(frame[1] >> 1) & 0x7, (frame[0] << 8 | frame[1]) & ~0x000E] += 1
    for vcid, packet in iter_packets(path):
        apid = packet_apid(packet)
        apids[apid] += 1
        if apid == APID:
            packets.append((path.name, vcid, packet))
        elif apid == 18 and packet[6:9] == TEXT_HEADER and len(packet) > 40:
            tick = int.from_bytes(packet[11:15])
            text_ticks[path.name].append(tick)
            text = packet[24:-5]
            if text.startswith(b"F4") and len(text) == 44:
                with suppress(ValueError):
                    gps_minus_tick.append(int(text[3:7]) * SECONDS_PER_WEEK + float(text[7:15]) - tick)
    print(f"processed {path.name}")

print(f"{len(packets)} packets with APID {APID}, of {sum(apids.values())} in total")
print("most common APIDs:", apids.most_common(12))
print("APIDs by packet count, the last ones:", sorted(apids.items(), key=lambda item: item[1])[:8])

processed proba2_dump(1).cadu
processed proba2_dump(10).cadu
processed proba2_dump(11).cadu
processed proba2_dump(12).cadu
processed proba2_dump(2).cadu
processed proba2_dump(3).cadu
processed proba2_dump(4).cadu
processed proba2_dump(5).cadu
processed proba2_dump(6).cadu
processed proba2_dump(7).cadu
processed proba2_dump(8).cadu
processed proba2_dump(9).cadu
processed proba2_dump.cadu
27 packets with APID 0, of 271472 in total
most common APIDs: [(2047, 228575), (2, 18399), (22, 11186), (18, 5025), (20, 4240), (14, 1738), (21, 697), (8, 599), (11, 131), (132, 92), (133, 91), (130, 58)]
APIDs by packet count, the last ones: [(1216, 1), (828, 1), (378, 1), (1844, 1), (1820, 1), (1668, 1), (1691, 1), (464, 1)]


## 2. Which virtual channels are real?

The packets of APID 0 sit on virtual channels 0, 1, 2 and 5. Before reading them: how many frames does each virtual
channel have, and do they all look like valid transfer frame headers?

In [5]:
frames_per_vc = Counter()
for (vcid, _), count in frame_headers.items():
    frames_per_vc[vcid] += count
print("frames per virtual channel:", dict(sorted(frames_per_vc.items())))
print("most common headers (virtual channel id cleared):")
for (vcid, header), count in frame_headers.most_common(8):
    print(f"  vc {vcid}  0x{header:04X}  {count}")
VALID_HEADER = frame_headers.most_common(1)[0][0][1]
for vcid, total in sorted(frames_per_vc.items()):
    share = frame_headers[vcid, VALID_HEADER] / total
    print(f"  vc {vcid}: {total:6d} frames, {share:6.1%} with the common header 0x{VALID_HEADER:04X}")

frames per virtual channel: {0: 1797, 1: 5452, 2: 11451, 3: 215246, 4: 31, 5: 68, 6: 30, 7: 29028}
most common headers (virtual channel id cleared):
  vc 3  0x1E51  215198
  vc 7  0x1E51  28998
  vc 2  0x1E51  11432
  vc 1  0x1E51  5417
  vc 0  0x1E51  1756
  vc 5  0x8F21  32
  vc 0  0x2650  9
  vc 3  0x1E50  8
  vc 0:   1797 frames,  97.7% with the common header 0x1E51
  vc 1:   5452 frames,  99.4% with the common header 0x1E51
  vc 2:  11451 frames,  99.8% with the common header 0x1E51
  vc 3: 215246 frames, 100.0% with the common header 0x1E51
  vc 4:     31 frames,   9.7% with the common header 0x1E51
  vc 5:     68 frames,   1.5% with the common header 0x1E51
  vc 6:     30 frames,  13.3% with the common header 0x1E51
  vc 7:  29028 frames,  99.9% with the common header 0x1E51


On virtual channels 0, 1, 2, 3 and 7 the great majority of the frames have the same header (transfer frame version
and spacecraft id), with 7 almost certainly the idle channel. On 4, 5 and 6 (129 frames in total) hardly any frame
has it: those are frames with bit errors (the dumps are not Reed-Solomon corrected) whose virtual channel id came
out wrong. So whatever was reassembled on virtual channels 4, 5 and 6 is not a packet.

## 3. The 27 packets

In [6]:
for name, vcid, packet in packets:
    flags = packet[2] >> 6
    print(
        f"{name:22s} vc {vcid}  flags {flags}  length {len(packet):5d}  "
        f"seq {packet_sequence_count(packet):5d}  {packet[:20].hex(' ')}"
    )

proba2_dump(1).cadu    vc 0  flags 3  length    15  seq  4092  00 00 cf fc 00 08 08 2f 0d f9 6a 22 ae b8 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  80 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 2  length     7  seq     0  00 00 80 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length  2164  seq     0  00 00 00 00 80 00 00 00 00 00 00 00 00 80 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  00 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  00 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  00 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  00 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  00 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  00 00 00 00 00 00 00
proba2_dump(1).cadu    vc 5  flags 0  length     7  seq     0  00 00 00 00 00 00 00
proba2_dump(1

In [7]:
groups = Counter((vcid, len(packet), packet[2] >> 6) for _, vcid, packet in packets)
print("(virtual channel, length, sequence flags):", dict(groups))

(virtual channel, length, sequence flags): {(0, 15, 3): 10, (5, 7, 0): 13, (5, 7, 2): 1, (5, 2164, 0): 1, (2, 7, 0): 1, (1, 129, 2): 1}


Three groups:

* 15 packets on virtual channel 5, 14 of them 7 bytes and one 2164 bytes, all with sequence flags 0 and zeros (or
  `0x80` every 9th byte) where the header should be: debris from the bit error frames above. Not packets.
* 1 packet of 129 bytes on virtual channel 1, which looks like a PUS housekeeping report (`10 03 19`) with APID 0
  (section 6 shows that it is an APID 2 report with bit errors in its header).
* 10 packets of 15 bytes, one in 10 of the 13 dumps, on virtual channel 0, with sequence flags 3 (unsegmented).
  These are the real ones.

## 4. The 15 byte packets

In [8]:
TIME_PACKET = 15


@dataclass
class TimePacket:
    file: str
    sequence: int
    constant: bytes  # payload bytes 0-1
    tick: int
    fraction: float
    last: int


time_packets = sorted(
    (
        TimePacket(
            name,
            packet_sequence_count(packet),
            packet[6:8],
            int.from_bytes(packet[8:12]),
            int.from_bytes(packet[12:14]) / 65536,
            packet[14],
        )
        for name, vcid, packet in packets
        if len(packet) == TIME_PACKET and vcid == 0
    ),
    key=lambda item: item.tick,
)
print("payload bytes 0-1:", Counter(item.constant.hex() for item in time_packets))
print("last byte:", Counter(item.last for item in time_packets))
print("primary header bytes 0-1:", Counter(packet[:2].hex() for _, _, packet in packets if len(packet) == TIME_PACKET))

payload bytes 0-1: Counter({'082f': 10})
last byte: Counter({0: 10})
primary header bytes 0-1: Counter({'0000': 10})


The payload is 2 constant bytes (`08 2f`), a 4 byte counter, a 2 byte fraction and a zero byte. The 4 byte counter
has the size and the values of the tick counter of the other APIDs. Compare.

In [9]:
print(f"{'file':24s} {'tick':>10s}  {'ticks of the APID 18 records in this file':s}")
for item in time_packets:
    ticks = np.array(text_ticks[item.file])
    ticks = ticks[abs(ticks - np.median(ticks)) < MAX_TICK_SPREAD] if len(ticks) else ticks
    span = (
        f"{ticks.min()} to {ticks.max()}  (packet is {item.tick - ticks.min():+d} s from the start)"
        if len(ticks)
        else "none"
    )
    print(f"{item.file:24s} {item.tick:10d}  {span}")

file                           tick  ticks of the APID 18 records in this file
proba2_dump(12).cadu      229046009  229045749 to 229046029  (packet is +260 s from the start)
proba2_dump(11).cadu      229218258  229218107 to 229218397  (packet is +151 s from the start)
proba2_dump(10).cadu      229611909  229605162 to 229612032  (packet is +6747 s from the start)
proba2_dump(9).cadu       229735368  229735280 to 229735510  (packet is +88 s from the start)
proba2_dump(8).cadu       229824469  229818729 to 229824579  (packet is +5740 s from the start)
proba2_dump(7).cadu       229907800  229907678 to 229907848  (packet is +122 s from the start)
proba2_dump(6).cadu       229996898  229991077 to 229996957  (packet is +5821 s from the start)
proba2_dump(4).cadu       230218110  230218044 to 230218114  (packet is +66 s from the start)
proba2_dump(1).cadu       234449442  234449432 to 234449582  (packet is +10 s from the start)
proba2_dump.cadu          234489946  234489904 to 234575976  (pack

Each packet sits inside the time span of its dump, so it is a time stamp in the same tick counter. Now the sequence
count: it is a counter of APID 0 packets, so if the counter goes up by one per packet, the packets come at a fixed
interval and this tells how long.

In [10]:
ticks = np.array([item.tick + item.fraction for item in time_packets])
sequences = np.array([item.sequence for item in time_packets])
gaps_in_time = np.diff(ticks)
gaps_in_count = np.diff(sequences) % SEQUENCE_MODULO

# The packets are far apart and the 14 bit count wraps, so a first estimate of the period from the closest pair decides
# how many wraps there are in a gap, then all the gaps together give the period.
first_estimate = gaps_in_time.min() / gaps_in_count[gaps_in_time.argmin()]
steps = gaps_in_count + SEQUENCE_MODULO * np.round((gaps_in_time / first_estimate - gaps_in_count) / SEQUENCE_MODULO)
print(f"packets between the dumps: {steps.astype(int)}")
print(f"period: {gaps_in_time.sum() / steps.sum():.2f} s; per gap: {np.round(gaps_in_time / steps, 2)}")

packets between the dumps: [  469  1071   335   243   227   242   602 11510   110]
period: 367.61 s; per gap: [367.27 367.55 368.53 366.67 367.1  368.17 367.46 367.62 368.22]


## 5. Dating them

The GPS time of the APID 18 `F4` records minus the tick is constant, so the packets can be dated.

In [11]:
offsets = np.array(gps_minus_tick)
offset = np.median(offsets)
low, high = np.percentile(offsets, [1, 99])
print(f"GPS seconds minus tick: median {offset:.1f}, 1 % to 99 %: {low:.1f} to {high:.1f} ({len(offsets)} records)")
for item in time_packets:
    when = GPS_EPOCH + timedelta(seconds=item.tick + item.fraction + offset)
    print(f"{item.file:24s} {when:%Y-%m-%d %H:%M:%S.%f}")

GPS seconds minus tick: median 1209613891.0, 1 % to 99 %: 1209613881.0 to 1209613946.2 (2453 records)
proba2_dump(12).cadu     2025-08-08 03:45:00.600098
proba2_dump(11).cadu     2025-08-10 03:35:49.794495
proba2_dump(10).cadu     2025-08-14 16:56:40.227966
proba2_dump(9).cadu      2025-08-16 03:14:19.449951
proba2_dump(8).cadu      2025-08-17 03:59:20.953247
proba2_dump(7).cadu      2025-08-18 03:08:11.817444
proba2_dump(6).cadu      2025-08-19 03:53:09.596130
proba2_dump(4).cadu      2025-08-21 17:20:01.358765
proba2_dump(1).cadu      2025-10-09 16:42:13.682495
proba2_dump.cadu         2025-10-10 03:57:17.366028


## 6. The 129 byte packet

In [12]:
(odd,) = (packet for _, vcid, packet in packets if len(packet) == 129)
print(odd.hex(" "))
print(
    f"secondary header {odd[6:9].hex(' ')}, bytes 9-16 {odd[9:17].hex(' ')}, "
    f"17-20 {odd[17:21].hex(' ')}, byte 21: {odd[21]}"
)
print(f"CRC-16 ok: {crc16(odd[:-2]) == int.from_bytes(odd[-2:])}")
print("float32 from byte 32:", np.round(np.frombuffer(odd[32:64], dtype=">f4"), 2))

08 00 84 f4 00 7a 10 03 19 33 45 d0 92 d7 38 19 f4 00 00 00 00 47 00 00 00 07 01 01 00 00 00 02 40 9e 5b 31 40 54 0e 39 41 10 9d 62 c1 10 9a 0b 42 a4 19 9a 43 93 d3 33 41 85 33 33 c1 6c ea 0f 41 06 2b 84 41 07 a1 dc 3e e2 13 79 43 a3 02 8f 43 87 ee 14 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 01 00 32 00 00 00 e0 00 00 10 d6 01 01 01 00 00 00 05 00 00 20 ba 00 00 00 00 a9 43
secondary header 10 03 19, bytes 9-16 33 45 d0 92 d7 38 19 f4, 17-20 00 00 00 00, byte 21: 71
CRC-16 ok: False
float32 from byte 32: [  4.95   3.31   9.04  -9.04  82.05 295.65  16.65 -14.81]


Compare with `APID_2.ipynb`: the secondary header `10 03 19` is the housekeeping report one, bytes 17-20 are the 4
zeros and byte 21 is the structure id, 71, and a report of 129 bytes with SID 71 is one of the common APID 2 layouts
(694 reports). From byte 32 the data reads as plausible float32 values. The primary header says APID 0 instead of 2 (`08 00`
against `08 02`), the marker and tick counter in bytes 9-16 are garbage, and the CRC fails. So this is an APID 2
report with bit errors in its header, not a packet of APID 0.

## 7. What we know so far

* APID 0 has 27 packets in the dumps. 15 are debris of bit error frames (virtual channels 4, 5, 6 have no valid
  frame headers; one more 7 byte packet is on channel 2), one is an APID 2 report whose header has bit errors.
* The 10 real ones are 15 byte time packets on virtual channel 0, one per dump. Payload: `08 2f`, the tick counter,
  a 2 byte fraction, `00`. They are a time reference on the same clock as the other APIDs (one tick is one second,
  GPS time dates it), sent at a fixed period (see section 4).
* No housekeeping, no measurements, no LYRA, nothing about the Sun or the attitude.